#### WRITE TO FUNCTION TO COMPUTE THE SOURCE AND TARGET ROW COUNTS MATCH

In [0]:
def verify_pipeline_counts(source_df, target_df, pipeline_name="ETL_Pipeline"):
    """
    Compares row counts between source and target DataFrames.
    Raises an AssertionError if they do not match.
    """
    print(f"[{pipeline_name}] Starting row count validation...")
    
    # 1. Trigger the counts (Note: count() is an action and will execute the Spark job)
    source_count = source_df.count()
    target_count = target_df.count()
    
    print(f"[{pipeline_name}] Source Row Count: {source_count}")
    print(f"[{pipeline_name}] Target Row Count: {target_count}")
    
    # 2. Formulate a detailed error message for debugging
    error_message = (
        f"Data mismatch detected in {pipeline_name}! "
        f"Source count ({source_count}) does not match Target count ({target_count}). "
        f"Difference: {abs(source_count - target_count)} rows."
    )
    
    # 3. Assert statement
    assert source_count == target_count, error_message
    
    print(f"[{pipeline_name}] Validation SUCCESS: Source and Target counts match perfectly.")
    return True

#### CREATE ROW FILTER FUNCTION FOR DATA SECURITY

In [0]:
%sql
CREATE OR REPLACE FUNCTION GIZMO.GOLD.filter_state_fn(state STRING)
RETURNS BOOLEAN
RETURN
    is_account_group_member('admins')
    OR (is_account_group_member('ELITE-STATE-SG') AND state IN ('Alaska', 'Arizona'))
    OR (is_account_group_member('NON-ELITE-STATE-SG') AND state NOT IN ('Alaska', 'Arizona'));

### CREATE COLUMN LEVEL MASKING

#### `GIZMO.GOLD.mask_billing_address_line_1_fn` - This function evaluates the user's group membership and determines wherther to include a a specific record in the output based on state

In [0]:
%sql
CREATE OR REPLACE FUNCTION GIZMO.GOLD.mask_billing_address_line_1_fn(billing_address_line_1 STRING)
RETURN
    CASE 
        WHEN is_account_group_member('admins') THEN billing_address_line_1
        ELSE '**************'
        END;

####`GIZMO.GOLD.email_fn` - This function evaluates the user's group membership and determines wherther to return the original value or masked version of the value

In [0]:
%sql
CREATE OR REPLACE FUNCTION GIZMO.GOLD.email_fn(email STRING)
RETURN
    CASE 
        WHEN is_account_group_member('admins') THEN email
        ELSE concat(substr(email, 1, 1), '****@g', split(email,'@')[1])
        END;